# Qa porosity reflection

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA: Porosity and Shoreline Reflection

This notebook validates static shoreline features from src/ray_caster.py by overlaying each site's 5 km porosity disk and nearest-shore normal vector on satellite imagery (EPSG:32633).

In [ ]:
from __future__ import annotations

from pathlib import Path

import contextily as ctx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage
from skimage.draw import disk

plt.rcParams["figure.dpi"] = 120
TARGET_EPSG = 32633
POROSITY_RADIUS_M = 5_000.0


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for parent in [cwd, *cwd.parents]:
        if all((parent / name).exists() for name in ("configs", "data", "notebooks", "src")):
            return parent
    raise FileNotFoundError("Could not resolve project root with configs/data/notebooks/src.")


PROJECT_ROOT = resolve_project_root()
RAY_CSV = PROJECT_ROOT / "data/processed/ray_features.csv"
BATHY_NPZ = PROJECT_ROOT / "data/processed/bathy/bathy_field_project_site.npz"

ray_df = pd.read_csv(RAY_CSV)
required = [
    "site_name",
    "site_x",
    "site_y",
    "site_row",
    "site_col",
    "static_porosity_5km",
    "static_nearest_shore_steepness",
    "static_nearest_shore_normal_deg",
]
missing = [col for col in required if col not in ray_df.columns]
if missing:
    raise KeyError(f"Missing required ray columns: {missing}")

site_static = ray_df.groupby("site_name", as_index=False).agg(
    site_x=("site_x", "first"),
    site_y=("site_y", "first"),
    site_row=("site_row", "first"),
    site_col=("site_col", "first"),
    static_porosity_5km=("static_porosity_5km", "first"),
    static_nearest_shore_steepness=("static_nearest_shore_steepness", "first"),
    static_nearest_shore_normal_deg=("static_nearest_shore_normal_deg", "first"),
)

site_static["site_row"] = pd.to_numeric(site_static["site_row"], errors="coerce").astype(int)
site_static["site_col"] = pd.to_numeric(site_static["site_col"], errors="coerce").astype(int)

with np.load(BATHY_NPZ, allow_pickle=True) as npz:
    x_coord = np.asarray(npz["x"], dtype=float)
    y_coord = np.asarray(npz["y"], dtype=float)
    land_mask = np.asarray(npz["land_mask"], dtype=bool)

dx = float(np.median(np.diff(x_coord))) if x_coord.size > 1 else 1.0
dy = float(np.median(np.diff(y_coord))) if y_coord.size > 1 else 1.0

water_mask = ~land_mask
_, nearest_land_idx = ndimage.distance_transform_edt(
    water_mask,
    sampling=(dy, dx),
    return_distances=True,
    return_indices=True,
)

site_rows = site_static["site_row"].to_numpy(dtype=int)
site_cols = site_static["site_col"].to_numpy(dtype=int)
shore_rows = nearest_land_idx[0, site_rows, site_cols].astype(int)
shore_cols = nearest_land_idx[1, site_rows, site_cols].astype(int)
site_static["shore_x"] = x_coord[shore_cols]
site_static["shore_y"] = y_coord[shore_rows]

print(f"Project root: {PROJECT_ROOT}")
print(f"Ray features: {RAY_CSV}")
print(f"Bathymetry grid: {BATHY_NPZ}")
print(f"Sites available: {len(site_static)}")

In [ ]:
selected_sites = site_static.head(8).copy()
if selected_sites.empty:
    raise ValueError("No sites available for plotting.")

fig, axes = plt.subplots(4, 2, figsize=(14, 14), squeeze=False)
axes = axes.ravel()

avg_spacing_m = max(1.0, 0.5 * (abs(dx) + abs(dy)))
radius_cells = max(1, int(np.round(POROSITY_RADIUS_M / avg_spacing_m)))

for ax, (_, site) in zip(axes, selected_sites.iterrows()):
    site_x = float(site["site_x"])
    site_y = float(site["site_y"])
    site_row = int(site["site_row"])
    site_col = int(site["site_col"])

    shore_x = float(site["shore_x"])
    shore_y = float(site["shore_y"])

    porosity = float(site["static_porosity_5km"])
    steepness = float(site["static_nearest_shore_steepness"])
    normal_deg = float(site["static_nearest_shore_normal_deg"])

    pad = POROSITY_RADIUS_M * 1.15
    ax.set_xlim(site_x - pad, site_x + pad)
    ax.set_ylim(site_y - pad, site_y + pad)

    try:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs="EPSG:32633")
    except Exception as exc:
        print(f"Basemap fetch failed ({exc}); plotting without imagery.")

    rr, cc = disk((site_row, site_col), radius=radius_cells, shape=land_mask.shape)
    ax.scatter(
        x_coord[cc],
        y_coord[rr],
        s=2.5,
        c="#ff4d4d",
        alpha=0.08,
        linewidths=0.0,
        zorder=2,
    )

    arrow_len_m = max(120.0, 2000.0 * steepness)
    theta = np.deg2rad(90.0 - normal_deg)
    arrow_dx = arrow_len_m * np.cos(theta)
    arrow_dy = arrow_len_m * np.sin(theta)

    ax.quiver(
        shore_x,
        shore_y,
        arrow_dx,
        arrow_dy,
        angles="xy",
        scale_units="xy",
        scale=1.0,
        color="#20f3ff",
        width=0.004,
        zorder=4,
    )

    ax.scatter(site_x, site_y, s=50, c="yellow", edgecolor="black", zorder=5)
    ax.scatter(shore_x, shore_y, s=30, c="#ffa500", edgecolor="black", zorder=5)

    ax.set_title(f"{site['site_name']}\nporosity={porosity:.3f}, shore_steepness={steepness:.4f}")
    ax.set_xlabel("X (m), EPSG:32633")
    ax.set_ylabel("Y (m), EPSG:32633")

for ax in axes[len(selected_sites) :]:
    ax.axis("off")

plt.tight_layout()
plt.show()